# 🛠️ Advanced Tool Use with GitHub Models (Python)

## 📋 Learning Objectives

This notebook demonstrates advanced tool integration patterns using the Microsoft Agent Framework with GitHub Models. You'll learn how to create, manage, and orchestrate multiple tools to build sophisticated agent capabilities.

**What You'll Master:**
- 🔧 **Multi-Tool Architecture**: Building agents with multiple specialized tools
- 🎯 **Tool Selection Logic**: How agents choose the right tool for each task
- 📊 **Data Processing Tools**: Creating tools that handle different data types
- 🔗 **Tool Composition**: Combining tools for complex workflows

## 🎯 Key Tool Patterns

### Tool Design Principles
- **Single Responsibility**: Each tool has a clear, focused purpose
- **Type Safety**: Strong typing for reliable tool execution
- **Error Handling**: Graceful failure and recovery patterns
- **Composability**: Tools that work well together

### Advanced Tool Features
- **Context Awareness**: Tools that understand conversation context
- **Data Validation**: Input sanitization and output validation
- **Performance Optimization**: efficient tool execution patterns
- **Extensibility**: Easy addition of new tool capabilities

## 🔧 Technical Architecture

### Core Components
- **Microsoft Agent Framework**: Python implementation with advanced tool support
- **GitHub Models Integration**: High-performance language model access
- **Tool Registry System**: Organized management of agent capabilities
- **Error Recovery Patterns**: Robust handling of tool execution failures

### Tool Integration Flow
```python
User Request → Agent Analysis → Tool Selection → Tool Execution → Response Synthesis
```

## 🛠️ Tool Categories Demonstrated

### 1. **Data Generation Tools**
- Random destination generator
- Weather information provider  
- Travel cost calculator
- Activity recommendation engine

### 2. **Processing Tools**
- Text formatting and validation
- Data transformation utilities
- Content analysis functions
- Response enhancement tools

### 3. **Integration Tools**
- External API connectors
- File system operations
- Database query interfaces
- Web scraping utilities

## ⚙️ Prerequisites & Setup


**Required Dependencies:**
```bash

pip install agent-framework-core -U
```

**Environment Configuration (.env file):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Optional APIs:**
- Weather service API key (for weather tools)
- Currency conversion API access
- Travel information service credentials

## 🎨 Design Patterns

### Tool Factory Pattern
- Centralized tool creation and configuration
- Consistent tool interface design
- Easy tool registration and discovery

### Command Pattern
- Encapsulated tool execution logic
- Undo/redo functionality for complex operations
- Audit logging for tool usage

### Observer Pattern
- Tool execution monitoring
- Performance metrics collection
- Error reporting and alerting

## 🚀 Best Practices

- **Tool Documentation**: Clear descriptions for agent understanding
- **Input Validation**: Robust parameter checking and sanitization
- **Output Formatting**: Consistent, parseable tool responses
- **Error Messages**: Helpful error information for debugging
- **Performance**: Optimized tool execution for responsiveness

Ready to build agents with powerful tool capabilities? Let's create something amazing! ⚡

In [ ]:
# ! pip install agent-framework-core -U

In [ ]:
# � Import core dependencies for Agent Framework and tool integration
# This sets up the essential libraries for building intelligent agents with tool capabilities
import os

from dotenv import load_dotenv  # For loading environment variables securely
from random import randint

from azure.ai.projects.aio import AIProjectClient  # Async client for Azure AI Projects
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity.aio import AzureCliCredential  # Async credential for Azure authentication

from pydantic import BaseModel, Field
from typing import Annotated, Literal, Optional, Dict, Any
from datetime import datetime, timezone

# from agent_framework import tool

In [ ]:
load_dotenv(override=True)

In [ ]:
# 🔑 Environment variables verification

print("AZURE_AI_PROJECT_ENDPOINT:", os.environ.get("AZURE_AI_PROJECT_ENDPOINT"))
print("AZURE_AI_MODEL_DEPLOYMENT_NAME:", os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME"))
print("AZURE_SEARCH_SERVICE_ENDPOINT:", os.environ.get("AZURE_SEARCH_SERVICE_ENDPOINT"))

In [ ]:
# # 🛠️ Define travel planning tools for agent integration
# # These functions provide specific capabilities that the agent can invoke dynamically

# def get_random_destination() -> str:
#     """
#     🎲 Random destination generator tool
#     Returns a randomly selected travel destination from curated list
#     Useful when customers need inspiration for their next vacation
#     """
#     destinations = [
#         "Paris, France",
#         "Tokyo, Japan", 
#         "New York City, USA",
#         "London, England",
#         "Rome, Italy",
#         "Sydney, Australia",
#         "Dubai, UAE",
#         "Barcelona, Spain",
#         "Bangkok, Thailand",
#         "Amsterdam, Netherlands",
#         "Istanbul, Turkey",
#         "Prague, Czech Republic",
#         "Santorini, Greece",
#         "Reykjavik, Iceland",
#         "Marrakech, Morocco",
#         "Cape Town, South Africa",
#         "Rio de Janeiro, Brazil",
#         "Bali, Indonesia"
#     ]
#     # 🎯 Return random selection from the curated destination list
#     return destinations[randint(0, len(destinations) - 1)]

In [ ]:
class WeatherResult(BaseModel):
    location: str
    temperature: float
    unit: str
    condition: str
    humidity_pct: int
    wind_kph: float
    observed_at_iso: str
    source: str

def get_weather_forecast(location: str) ->  Dict[str, Any]:
    """
    ☀️ Weather forecast tool
    Provides a mock weather forecast for a given location.
    """
    # In a real implementation, this would call a weather API
    # Here we return a static mock response for demonstration purposes

    result = WeatherResult(
    location=location,
    temperature=60,
    unit="F",
    condition="Sunny",
    humidity_pct=60,
    wind_kph=12.5,
    observed_at_iso=datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    source="demo",
)
    return result.model_dump()

In [ ]:
# 🤖 Configure weather agent identity and behavioral instructions
# Define the agent's personality, capabilities, and operational guidelines

AGENT_NAME = "WeatherAgent"

AGENT_INSTRUCTIONS = "You are a helpful AI Agent that can provide weather forecasts for various locations."

In [ ]:
# create and interact with an agent

async with (
    AzureCliCredential() as credential,
    AzureAIProjectAgentProvider(project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"], 
                                credential=credential) as provider,
):
    # First, create an agent using the SDK directly
    agent = await provider.create_agent(
        name=AGENT_NAME,
        description="Weather Agent.",
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions=AGENT_INSTRUCTIONS,
        tools=[get_weather_forecast],
        )
    
    agent_name = agent.name

    thread = agent.get_new_thread() # Create a new conversation thread for the agent to interact within

    query = "Get the weather forecast for Washington, DC"
    print(f"User: {query}")

    result = await agent.run(query, thread=thread)
    print(f"Agent: {result.text}")

    print(f"agent id: {agent.id}")

In [ ]:
# Retrieve an existing agent by name and interact with it

async with (
    AzureCliCredential() as credential,
    AzureAIProjectAgentProvider(project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"], 
                                credential=credential) as provider,
):

    agent = await provider.get_agent(
        name=AGENT_NAME,
        tools=[get_weather_forecast],
        )
    
    thread = agent.get_new_thread()

    query = "Get the weather forecast for Reston, VA"
    print(f"User: {query}")

    result = await agent.run(query, thread=thread)
    print(f"Agent: {result.text}")

    print(f"agent id: {agent.id}")

In [ ]:
# Delete Agent

async with (
    AzureCliCredential() as credential,
    AIProjectClient(endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"], credential=credential) as project_client,
):
    await project_client.agents.delete(
        agent_name=agent_name
    )

In [ ]:
from agent_framework import ChatResponse, HostedCodeInterpreterTool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity.aio import AzureCliCredential
from openai.types.responses.response import Response as OpenAIResponse
from openai.types.responses.response_code_interpreter_tool_call import ResponseCodeInterpreterToolCall

In [ ]:
async with (
    AzureCliCredential() as credential,
    AzureAIProjectAgentProvider(credential=credential) as provider,
):
    agent = await provider.create_agent(
        name="MyCodeInterpreterAgent",
        instructions="You are a helpful assistant that can write and execute Python code to solve problems.",
        tools=HostedCodeInterpreterTool(),
    )

    query = f"Use code to get the factorial of 100?"
    print(f"User: {query}")
    result = await agent.run(query)
    print(f"Result: {result}\n")

    if (
        isinstance(result.raw_representation, ChatResponse)
        and isinstance(result.raw_representation.raw_representation, OpenAIResponse)
        and len(result.raw_representation.raw_representation.output) > 0
    ):
        # Find the first ResponseCodeInterpreterToolCall item
        code_interpreter_item = next(
            (
                item
                for item in result.raw_representation.raw_representation.output
                if isinstance(item, ResponseCodeInterpreterToolCall)
            ),
            None,
        )

        if code_interpreter_item is not None:
            generated_code = code_interpreter_item.code
            print(f"Generated code:\n{generated_code}")
